# CSE457: XỬ LÝ ÂM THANH VÀ TIẾNG NÓI
## BÀI THỰC HÀNH 1: PHÂN TÍCH VÀ XỬ LÝ TÍN HIỆU ÂM THANH SỐ
**Chủ đề:** Từ tín hiệu rời rạc đến FFT/STFT, Lọc số, Lượng tử hóa và Mã hóa âm thanh  
**Môi trường:** Python 3.x (NumPy, SciPy, Matplotlib)

---
### Mục tiêu học phần:
1. Đọc và phân tích siêu dữ liệu âm thanh số ($F_s$, kênh, bit depth, duration).
2. Trích xuất đặc trưng miền thời gian: Waveform, Peak, RMS (dBFS), Năng lượng.
3. Biến đổi miền tần số bằng FFT: Độ phân giải bin $\Delta f$, Peak Harmonics, vai trò của Zero-Padding.
4. Phân tích thời gian - tần số với STFT & Spectrogram: Đánh đổi Heisenberg-Gabor qua các độ dài khung.
5. So sánh hiệu ứng cửa sổ (Windowing): Rectangular vs. Hamming, hiện tượng rò rỉ phổ (Spectral Leakage).
6. Thiết kế và ứng dụng bộ lọc số FIR pha tuyến tính (Low-pass, High-pass, Band-pass) và khảo sát độ trễ nhóm.
7. Thực nghiệm lượng tử hóa đều $B$-bit, đo đạc SNR thực nghiệm đối chiếu lý thuyết, Resampling chống chồng phổ và phân tích tỷ số nén (Compression Ratio).


## 0. Cài đặt môi trường và Thư viện cần thiết
Các thư viện sử dụng:
- `numpy`: Xử lý mảng đa chiều và biến đổi số học.
- `scipy.io.wavfile`: Đọc và ghi tệp âm thanh dạng WAV.
- `scipy.signal`: Thiết kế bộ lọc số (firwin, freqz, lfilter), tính STFT/Spectrogram và Resampling.
- `matplotlib.pyplot`: Trực quan hóa dữ liệu và xuất đồ thị kỹ thuật.


In [ ]:
import os
import numpy as np
from scipy import signal
from scipy.io import wavfile
import matplotlib.pyplot as plt

# Thiết lập hiển thị đồ thị đẹp mắt, sắc nét
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120

# Đường dẫn thư mục lưu trữ sản phẩm
AUDIO_DIR = "audio"
FIGURES_DIR = "figures"
os.makedirs(AUDIO_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

print("Đã nạp đầy đủ các thư viện và khởi tạo môi trường thực hành.")


---
## Khối A: Đọc và Kiểm tra Dữ liệu Âm thanh (Metadata & Preprocessing)

### Cơ sở lý thuyết:
- **Tần số lấy mẫu ($F_s$):** Số lượng mẫu tín hiệu rời rạc được thu thập trong mỗi giây ($1/T$). Tần số Nyquist tương ứng là $F_{Nyquist} = F_s / 2$.
- **Kênh âm thanh (Channels):** Stereo gồm 2 kênh độc lập (Trái/Phải). Để thuận tiện cho phân tích toán học 1 chiều, ta tính trung bình:
  $$x_{mono}[n] = \frac{x_L[n] + x_R[n]}{2}$$
- **Chuẩn hóa biên độ:** Dữ liệu 16-bit nguyên có dải giá trị $[-32768, 32767]$. Ta chuẩn hóa về số thực dấu phẩy động trong khoảng $[-1.0, 1.0]$ bằng cách chia cho $2^{B-1} = 32768.0$.


In [ ]:
# Đọc tệp âm thanh đầu vào
input_file = os.path.join(AUDIO_DIR, "input.wav")
Fs, raw_audio = wavfile.read(input_file)

# Trích xuất các thuộc tính siêu dữ liệu (Metadata)
file_size_bytes = os.path.getsize(input_file)
n_samples = raw_audio.shape[0]
n_channels = 1 if raw_audio.ndim == 1 else raw_audio.shape[1]
duration = n_samples / Fs
bit_depth = 16 if raw_audio.dtype == np.int16 else 32

print(f"--- THÔNG TIN TỆP ÂM THANH: {os.path.basename(input_file)} ---")
print(f"1. Tần số lấy mẫu (Fs)   : {Fs:,} Hz (Nyquist = {Fs/2:,} Hz)")
print(f"2. Số lượng kênh         : {n_channels} ({'Stereo' if n_channels == 2 else 'Mono'})")
print(f"3. Tổng số mẫu (N)       : {n_samples:,} samples")
print(f"4. Thời lượng tín hiệu   : {duration:.3f} s")
print(f"5. Định dạng dữ liệu     : {raw_audio.dtype} ({bit_depth} bit/sample)")
print(f"6. Kích thước file vật lý: {file_size_bytes:,} bytes (~{file_size_bytes / (1024*1024):.2f} MB)")

# Chuyển đổi về Mono và chuẩn hóa số thực về [-1.0, 1.0]
if n_channels == 2:
    left = raw_audio[:, 0].astype(np.float64) / (2**(bit_depth - 1))
    right = raw_audio[:, 1].astype(np.float64) / (2**(bit_depth - 1))
    x = (left + right) / 2.0
else:
    x = raw_audio.astype(np.float64) / (2**(bit_depth - 1))

print("\n-> Đã chuyển tín hiệu về mono và chuẩn hóa về miền [-1.0, 1.0].")


---
## Khối B: Phân tích Miền Thời gian (Time-Domain Analysis)

### Cơ sở lý thuyết:
- **Biên độ đỉnh (Peak):** $\text{Peak} = \max_n |x[n]|$. Dùng để phát hiện nguy cơ xén ngọn (clipping) khi Peak tiệm cận $1.0$ ($0$ dBFS).
- **Năng lượng hiệu dụng (RMS):** 
  $$\text{RMS} = \sqrt{\frac{1}{N} \sum_{n=0}^{N-1} x^2[n]}, \quad \text{RMS}_{\text{dBFS}} = 20 \log_{10}(\text{RMS})$$
- **Tổng năng lượng:** $E = \sum_{n=0}^{N-1} x^2[n]$.


In [ ]:
# Tính toán các chỉ số thống kê năng lượng trên toàn bộ tín hiệu
peak_val = np.max(np.abs(x))
rms_val = np.sqrt(np.mean(x**2))
rms_dbfs = 20 * np.log10(rms_val + 1e-12)
total_energy = np.sum(x**2)

print(f"Peak biên độ toàn bài  : {peak_val:.4f} ({'An toàn, không bị clipping' if peak_val < 0.99 else 'CẢNH BÁO CLIPPING!'})")
print(f"RMS toàn bài           : {rms_val:.4f} ({rms_dbfs:.2f} dBFS)")
print(f"Tổng năng lượng (E)    : {total_energy:.2f}")

# So sánh 2 phân đoạn khác biệt: Đoạn 1 (Hòa âm cao trào 1-2s) vs Đoạn 2 (Khoảng lặng 8.5-9.5s)
seg1 = x[int(1.0 * Fs):int(2.0 * Fs)]
seg2 = x[int(8.5 * Fs):int(9.5 * Fs)]

peak1, rms1 = np.max(np.abs(seg1)), np.sqrt(np.mean(seg1**2))
peak2, rms2 = np.max(np.abs(seg2)), np.sqrt(np.mean(seg2**2))

print(f"\n--- SO SÁNH 2 PHÂN ĐOẠN ---")
print(f"• Đoạn 1 (Hòa âm 1-2s)  : Peak = {peak1:.4f}, RMS = {rms1:.4f} ({20*np.log10(rms1):.2f} dBFS)")
print(f"• Đoạn 2 (Lặng 8.5-9.5s): Peak = {peak2:.4f}, RMS = {rms2:.4f} ({20*np.log10(rms2):.2f} dBFS)")
print(f"-> Tỷ số năng lượng RMS: Đoạn 1 gấp {rms1/rms2:.1f} lần Đoạn 2.")

# Vẽ đồ thị Waveform
t_axis = np.linspace(0, duration, len(x))
fig, axes = plt.subplots(3, 1, figsize=(12, 7))

axes[0].plot(t_axis, x, color='#1f77b4', lw=0.6)
axes[0].axvspan(1.0, 2.0, color='orange', alpha=0.3, label='Đoạn 1 (Hòa âm)')
axes[0].axvspan(8.5, 9.5, color='green', alpha=0.3, label='Đoạn 2 (Khoảng lặng)')
axes[0].set_title(f"Waveform toàn bài (Peak = {peak_val:.3f}, RMS = {rms_val:.3f})", fontweight='bold')
axes[0].set_ylabel("Biên độ")
axes[0].legend(loc='upper right')

# Zoom đoạn 0.5s (1.0s đến 1.5s)
zoom_t = np.linspace(1.0, 1.5, int(0.5 * Fs))
axes[1].plot(zoom_t, x[int(1.0*Fs):int(1.5*Fs)], color='#d62728', lw=0.8)
axes[1].set_title("Phân đoạn phóng to 0.5 giây (1.0s - 1.5s)", fontweight='bold')
axes[1].set_ylabel("Biên độ")

# So sánh Đoạn 1 vs Đoạn 2
t_loc = np.linspace(0, 1.0, len(seg1))
axes[2].plot(t_loc, seg1, color='orange', lw=0.8, label=f'Đoạn 1 (RMS = {rms1:.3f})')
axes[2].plot(t_loc, seg2, color='green', lw=0.8, alpha=0.8, label=f'Đoạn 2 (RMS = {rms2:.3f})')
axes[2].set_title("So sánh trực tiếp: Đoạn tín hiệu thực vs Khoảng lặng", fontweight='bold')
axes[2].set_xlabel("Thời gian (giây)")
axes[2].set_ylabel("Biên độ")
axes[2].legend(loc='upper right')

plt.tight_layout()
plt.show()


---
## Khối C: Phân tích Miền Tần số bằng FFT (Frequency-Domain Analysis)

### Cơ sở lý thuyết:
- **Biến đổi Fourier Rời rạc (DFT):**
  $$X[k] = \sum_{n=0}^{N-1} x[n] e^{-j \frac{2\pi k n}{N}}, \quad k = 0, \dots, N-1$$
- **Quy đổi tần số & Khoảng cách Bin:** Tần số vật lý tại bin $k$ là $f_k = k \frac{F_s}{\text{NFFT}}$, với độ rộng bin $\Delta f = \frac{F_s}{\text{NFFT}}$.
- **Zero-Padding:** Thêm số 0 vào đuôi tín hiệu để tăng NFFT làm các điểm mẫu trên đồ thị dày hơn (nội suy phổ sinc) nhưng **không làm tăng độ phân giải vật lý thực tế**.


In [ ]:
# Chọn đoạn ổn định 1.0s và nhân cửa sổ Hamming
fft_seg = x[int(1.0 * Fs):int(2.0 * Fs)]
w_hamm = np.hamming(len(fft_seg))
x_win = fft_seg * w_hamm

# 1. Tính FFT với NFFT lớn (65536)
NFFT_large = 65536
X_large = np.fft.rfft(x_win, n=NFFT_large)
freqs_large = np.fft.rfftfreq(NFFT_large, d=1.0/Fs)
mag_db_large = 20 * np.log10(np.maximum(np.abs(X_large), 1e-12))
mag_db_large -= np.max(mag_db_large)  # chuẩn hóa đỉnh về 0 dB

delta_f_large = Fs / NFFT_large

# 2. Tìm ít nhất 3 đỉnh phổ nổi bật
search_idx = np.where((freqs_large >= 100) & (freqs_large <= 3000))[0]
pks, _ = signal.find_peaks(mag_db_large[search_idx], height=-25, distance=int(50/delta_f_large))
top_pks = search_idx[pks][np.argsort(mag_db_large[search_idx[pks]])[::-1][:5]]
top_pks = np.sort(top_pks)

print(f"Cấu hình FFT lớn: NFFT = {NFFT_large}, Khoảng cách bin Delta_f = {delta_f_large:.4f} Hz")
print("Các đỉnh phổ nổi bật tìm thấy:")
for i, idx in enumerate(top_pks, 1):
    print(f"  Đỉnh {i}: Tần số = {freqs_large[idx]:7.2f} Hz, Biên độ = {mag_db_large[idx]:6.2f} dB")

# 3. Tính FFT với NFFT nhỏ (2048) để đối chiếu
NFFT_small = 2048
X_small = np.fft.rfft(x_win[:NFFT_small] * np.hamming(NFFT_small), n=NFFT_small)
freqs_small = np.fft.rfftfreq(NFFT_small, d=1.0/Fs)
mag_db_small = 20 * np.log10(np.maximum(np.abs(X_small), 1e-12))
mag_db_small -= np.max(mag_db_small)
delta_f_small = Fs / NFFT_small

# Vẽ đồ thị phổ
fig, axes = plt.subplots(2, 1, figsize=(12, 7))

axes[0].plot(freqs_large, mag_db_large, color='#1f77b4', lw=0.9)
for idx in top_pks:
    axes[0].plot(freqs_large[idx], mag_db_large[idx], 'r*', markersize=9)
    axes[0].annotate(f"{freqs_large[idx]:.1f} Hz", (freqs_large[idx]+30, mag_db_large[idx]+2), fontsize=8, color='red')
axes[0].set_title(f"Phổ FFT (Hamming window, NFFT = {NFFT_large})", fontweight='bold')
axes[0].set_ylabel("Biên độ tương đối (dB)")
axes[0].set_xlim(0, 4000)
axes[0].set_ylim(-70, 5)

axes[1].plot(freqs_small, mag_db_small, 'o-', color='orange', markersize=3, lw=0.9, label=f'NFFT nhỏ = {NFFT_small} (Delta_f = {delta_f_small:.2f} Hz)')
axes[1].plot(freqs_large, mag_db_large, color='#1f77b4', lw=0.8, alpha=0.9, label=f'NFFT lớn = {NFFT_large} (Delta_f = {delta_f_large:.2f} Hz)')
axes[1].set_title("So sánh độ phân giải lưới tần số giữa NFFT nhỏ vs NFFT lớn", fontweight='bold')
axes[1].set_xlabel("Tần số (Hz)")
axes[1].set_ylabel("Biên độ (dB)")
axes[1].set_xlim(0, 2000)
axes[1].set_ylim(-60, 5)
axes[1].legend(loc='upper right')

plt.tight_layout()
plt.show()


---
## Khối D: STFT và Spectrogram (Time-Frequency Analysis)

### Cơ sở lý thuyết:
- **Biến đổi Fourier Ngắn hạn (STFT):**
  $$X[m, k] = \sum_{n} x[n] w[n - mH] e^{-j \frac{2\pi k n}{\text{NFFT}}}$$
  Trong đó: $L$ là frame length, $H$ là hop size (bước trượt giữa các khung).
- **Đánh đổi Thời gian - Tần số (Heisenberg-Gabor Trade-off):** $\Delta t \cdot \Delta f \ge \frac{1}{4\pi}$.
  - Frame ngắn (10 ms): Độ phân giải thời gian cao (thấy rõ xung gõ nhịp transient), nhưng dải tần số bị nhòe rộng.
  - Frame dài (50 ms): Độ phân giải tần số cao (thấy rõ từng vạch họa âm mỏng sắc nét), nhưng sự kiện thời gian bị mờ nhòe.


In [ ]:
# 1. Spectrogram cấu hình chuẩn (L = 25 ms, H = 10 ms)
L_std = int(round(0.025 * Fs))  # 1102 mẫu
H_std = int(round(0.010 * Fs))  # 441 mẫu
NFFT_stft = 4096

f_s, t_s, S_s = signal.spectrogram(x, fs=Fs, window='hamming', nperseg=L_std, noverlap=L_std-H_std, nfft=NFFT_stft, mode='magnitude')
S_db = 20 * np.log10(np.maximum(S_s, 1e-6))
vmin, vmax = np.percentile(S_db, 15), np.max(S_db)

f_mask = f_s <= 6000

plt.figure(figsize=(12, 4))
plt.pcolormesh(t_s, f_s[f_mask], S_db[f_mask, :], shading='auto', cmap='magma', vmin=vmin, vmax=vmax)
plt.colorbar(label='Biên độ phổ (dB)')
plt.title(f"Spectrogram chuẩn (Khung 25 ms, Hop 10 ms, Overlap 60%)", fontweight='bold')
plt.xlabel("Thời gian (giây)")
plt.ylabel("Tần số (Hz)")
plt.tight_layout()
plt.show()

# 2. Thí nghiệm so sánh 3 cấu hình Frame Length: 10 ms, 25 ms, 50 ms
configs = [("10 ms (Ưu tiên Thời gian)", 0.010), ("25 ms (Cân bằng Chuẩn)", 0.025), ("50 ms (Ưu tiên Tần số)", 0.050)]
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True, sharey=True)

for i, (name, dur) in enumerate(configs):
    L_c = int(round(dur * Fs))
    H_c = int(round(0.010 * Fs))
    f_c, t_c, S_c = signal.spectrogram(x, fs=Fs, window='hamming', nperseg=L_c, noverlap=L_c-H_c, nfft=NFFT_stft, mode='magnitude')
    S_c_db = 20 * np.log10(np.maximum(S_c, 1e-6))
    f_m = f_c <= 5000
    im = axes[i].pcolormesh(t_c, f_c[f_m], S_c_db[f_m, :], shading='auto', cmap='magma', vmin=vmin, vmax=vmax)
    axes[i].set_title(f"{name} - L = {L_c} mẫu ({dur*1000:.0f} ms)", fontweight='bold')
    axes[i].set_ylabel("Tần số (Hz)")

axes[2].set_xlabel("Thời gian (giây)")
fig.subplots_adjust(right=0.88)
cbar_ax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
fig.colorbar(im, cax=cbar_ax, label='Biên độ phổ (dB)')
plt.show()


---
## Khối E: Thí nghiệm Cửa sổ (Windowing Experiment)

### Cơ sở lý thuyết:
- **Cửa sổ Chữ nhật (Rectangular):** Búp chính (Main-lobe) hẹp ($\approx 4\pi/L$), nhưng búp phụ (Side-lobes) cao (chỉ suy hao $-13$ dB), dẫn đến năng lượng bị tràn sang các tần số lân cận $\rightarrow$ **Rò rỉ phổ (Spectral Leakage)** nghiêm trọng.
- **Cửa sổ Hamming:** Búp phụ bị nén sâu xuống $\approx -41$ dB (triệt tiêu rò rỉ phổ cực tốt). Tuy nhiên búp chính bị bè rộng gấp đôi ($\approx 8\pi/L$).


In [ ]:
# Cắt 1 khung tín hiệu cố định 25ms
frame = x[int(1.2 * Fs):int(1.2 * Fs + L_std)]
N_f = len(frame)

w_rect = np.ones(N_f)
w_hamm = np.hamming(N_f)

NFFT_w = 8192
X_r = np.fft.rfft(frame * w_rect, n=NFFT_w)
X_h = np.fft.rfft(frame * w_hamm, n=NFFT_w)
f_w = np.fft.rfftfreq(NFFT_w, d=1.0/Fs)

db_r = 20 * np.log10(np.maximum(np.abs(X_r), 1e-12))
db_h = 20 * np.log10(np.maximum(np.abs(X_h), 1e-12))
db_r -= np.max(db_r)
db_h -= np.max(db_h)

plt.figure(figsize=(12, 4))
plt.plot(f_w, db_r, color='red', lw=0.9, alpha=0.7, label='Cửa sổ Chữ nhật (Rectangular) - Búp phụ cao (-13 dB), rò rỉ phổ mạnh')
plt.plot(f_w, db_h, color='#1f77b4', lw=1.2, label='Cửa sổ Hamming - Búp phụ thấp (-41 dB), triệt tiêu rò rỉ phổ')
plt.title(f"So sánh phổ logarit giữa Cửa sổ Chữ nhật và Hamming trên cùng một khung tín hiệu", fontweight='bold')
plt.xlabel("Tần số (Hz)")
plt.ylabel("Biên độ chuẩn hóa (dB)")
plt.xlim(0, 3000)
plt.ylim(-70, 5)
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()


---
## Khối F: Thiết kế Bộ lọc số FIR và Lọc Tín hiệu (Digital Filtering)

### Cơ sở lý thuyết:
- **Bộ lọc FIR đối xứng:** Có pha tuyến tính (Linear Phase) lý tưởng, không gây méo pha dạng sóng.
- **Độ trễ nhóm (Group Delay):** Cố định cho mọi tần số:
  $$\tau = \frac{\text{numtaps} - 1}{2} \text{ mẫu} = \frac{\text{numtaps} - 1}{2 F_s} \text{ giây}$$
  Với 201 taps ở 44.1 kHz, $\tau = 100 \text{ mẫu} \approx 2.27$ ms (rất nhỏ, hoàn toàn chấp nhận được trong xử lý thời gian thực).


In [ ]:
numtaps = 201
delay_samples = (numtaps - 1) // 2
delay_ms = (delay_samples / Fs) * 1000

print(f"Số hệ số lọc FIR : {numtaps} taps (Bậc M = {numtaps - 1})")
print(f"Độ trễ nhóm      : {delay_samples} mẫu ({delay_ms:.2f} ms)")

# 1. Thiết kế bộ lọc: Low-pass (2000 Hz), High-pass (3000 Hz), Band-pass (1000 - 3000 Hz)
b_lpf = signal.firwin(numtaps, cutoff=2000, fs=Fs, window='hamming', pass_zero='lowpass')
b_hpf = signal.firwin(numtaps, cutoff=3000, fs=Fs, window='hamming', pass_zero='highpass')
b_bpf = signal.firwin(numtaps, cutoff=[1000, 3000], fs=Fs, window='hamming', pass_zero='bandpass')

# 2. Tính đáp ứng tần số
w_l, H_l = signal.freqz(b_lpf, worN=4096, fs=Fs)
w_h, H_h = signal.freqz(b_hpf, worN=4096, fs=Fs)
w_b, H_b = signal.freqz(b_bpf, worN=4096, fs=Fs)

plt.figure(figsize=(12, 4))
plt.plot(w_l, 20*np.log10(np.maximum(np.abs(H_l), 1e-6)), color='#1f77b4', lw=1.5, label='Low-pass (fc = 2000 Hz)')
plt.plot(w_h, 20*np.log10(np.maximum(np.abs(H_h), 1e-6)), color='#d62728', lw=1.5, label='High-pass (fc = 3000 Hz)')
plt.plot(w_b, 20*np.log10(np.maximum(np.abs(H_b), 1e-6)), color='#2ca02c', lw=1.5, label='Band-pass (1000 - 3000 Hz)')
plt.axhline(-3, color='black', linestyle='--', lw=0.8, label='Mức suy hao -3 dB')
plt.title(f"Đáp ứng tần số |H(f)| của các bộ lọc FIR (Hamming window, {numtaps} taps)", fontweight='bold')
plt.xlabel("Tần số (Hz)")
plt.ylabel("Độ lợi (dB)")
plt.xlim(0, 6000)
plt.ylim(-80, 5)
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

# 3. Lọc tín hiệu bằng lfilter và so sánh phổ trước/sau
y_lpf = signal.lfilter(b_lpf, [1.0], x)

seg_raw = x[int(1.0*Fs):int(2.0*Fs)] * np.hamming(Fs)
seg_flt = y_lpf[int(1.0*Fs):int(2.0*Fs)] * np.hamming(Fs)

X_raw = np.fft.rfft(seg_raw, n=16384)
X_flt = np.fft.rfft(seg_flt, n=16384)
f_ax = np.fft.rfftfreq(16384, d=1.0/Fs)

plt.figure(figsize=(12, 4))
plt.plot(f_ax, 20*np.log10(np.maximum(np.abs(X_raw), 1e-12)), color='#1f77b4', lw=0.9, alpha=0.7, label='Phổ trước khi lọc (Gốc)')
plt.plot(f_ax, 20*np.log10(np.maximum(np.abs(X_flt), 1e-12)), color='#d62728', lw=1.1, label='Phổ sau khi lọc Low-pass 2000 Hz')
plt.axvline(2000, color='black', linestyle=':', label='Tần số cắt fc = 2000 Hz')
plt.title("Phổ trước và sau khi qua bộ lọc Low-pass FIR (Dải tần > 2000 Hz bị suy hao mạnh)", fontweight='bold')
plt.xlabel("Tần số (Hz)")
plt.ylabel("Biên độ (dB)")
plt.xlim(0, 6000)
plt.ylim(-65, 30)
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()


---
## Khối G: Lượng tử hóa, Resampling và Mã hóa Âm thanh

### Cơ sở lý thuyết:
- **Lượng tử hóa đều $B$-bit:** Bước lượng tử $\Delta = \frac{2 X_{max}}{2^B}$.
- **Tỷ số SNR lý thuyết:**
  $$\text{SNR}_{\text{theory}} = 6.02 B + 4.77 - 20 \log_{10}\left(\frac{X_{max}}{\sigma_x}\right) \quad (\text{dB})$$
- **Tốc độ bit & Tỷ số nén:**
  $$R_{PCM} = F_s \times B \times C \quad (\text{bps}), \quad \text{Compression Ratio} = \frac{R_{PCM}}{R_{compressed}}$$


In [ ]:
def uniform_quantize(sig, B):
    qmax = 2**(B - 1) - 1
    return np.round(np.clip(sig, -1.0, 1.0) * qmax) / qmax

def calc_snr(orig, quant):
    err = quant - orig
    return 10.0 * np.log10(np.sum(orig**2) / np.sum(err**2))

bits = [4, 6, 8, 12, 16]
snr_m, snr_t = [], []
sigma_x = np.std(x)
crest_term = 20 * np.log10(1.0 / sigma_x)

print(f"Độ lệch chuẩn sigma_x = {sigma_x:.4f}, Số hạng hệ số đỉnh = {crest_term:.2f} dB\n")
print(f"{'Bit (B)':<8} | {'SNR Đo được (dB)':<18} | {'SNR Lý thuyết (dB)':<20} | {'Sai lệch (dB)':<12}")
print("-" * 65)

for B in bits:
    xq = uniform_quantize(x, B)
    m = calc_snr(x, xq)
    t = 6.02 * B + 4.77 - crest_term
    snr_m.append(m)
    snr_t.append(t)
    print(f"{B:<8} | {m:<18.2f} | {t:<20.2f} | {abs(m-t):<12.2f}")

plt.figure(figsize=(9, 4))
plt.plot(bits, snr_m, 'o-', color='#1f77b4', lw=1.5, markersize=7, label='SNR Đo được thực nghiệm')
plt.plot(bits, snr_t, 's--', color='#d62728', lw=1.2, markersize=6, label=r'SNR Lý thuyết ($6.02B + 4.77 - 20\log_{10}(X_{max}/\sigma_x)$)')
plt.title("Tỷ số SNR lượng tử hóa thực nghiệm đối chiếu lý thuyết", fontweight='bold')
plt.xlabel("Số bit trên mẫu (B)")
plt.ylabel("SNR (dB)")
plt.xticks(bits)
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()

# So sánh tốc độ bit và tỷ số nén
codecs = [
    ("PCM 16-bit Stereo", 1411.2),
    ("MP3 256 kbps", 256.0),
    ("MP3 128 kbps", 128.0),
    ("Opus 64 kbps", 64.0)
]

print("\n--- BẢNG SO SÁNH TỐC ĐỘ BIT VÀ TỶ SỐ NÉN ---")
print(f"{'Codec / Định dạng':<22} | {'Bitrate (kbps)':<15} | {'Tỷ số nén':<12} | {'Tiết kiệm (%)':<14}")
print("-" * 70)
for name, br in codecs:
    ratio = 1411.2 / br
    sav = (1.0 - br / 1411.2) * 100.0
    print(f"{name:<22} | {br:<15.1f} | {ratio:<12.2f}x | {sav:<14.1f}%")

plt.figure(figsize=(9, 4))
bars = plt.bar([c[0] for c in codecs], [c[1] for c in codecs], color=['#1f77b4', '#2ca02c', '#ff7f0e', '#d62728'], width=0.5)
for bar in bars:
    y = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, y + 25, f"{y:.1f} kbps", ha='center', fontweight='bold', fontsize=9)
plt.title("So sánh tốc độ bit PCM và các định dạng nén âm thanh", fontweight='bold')
plt.ylabel("Tốc độ bit (kbps)")
plt.ylim(0, 1600)
plt.tight_layout()
plt.show()


---
## Kết luận và Đánh giá
Qua bài thực hành Lab 1, chúng ta đã kiểm chứng thực nghiệm đầy đủ các nền tảng cốt lõi của xử lý tín hiệu âm thanh số:
1. **Miền thời gian:** Phân biệt rõ sự khác biệt giữa đoạn năng lượng cao (hòa âm) và đoạn tĩnh/lặng qua Peak, RMS và Energy.
2. **Miền tần số (FFT):** Hiểu rõ bản chất của khoảng cách bin $\Delta f$ và phân biệt với độ phân giải vật lý thực sự.
3. **STFT & Spectrogram:** Quan sát trực quan sự đánh đổi Heisenberg-Gabor giữa độ phân giải thời gian và tần số khi thay đổi độ dài khung $L$.
4. **Cửa sổ hóa:** Minh chứng rõ ràng cửa sổ Hamming nén búp phụ xuống $-41$ dB để triệt tiêu rò rỉ phổ so với mức $-13$ dB của cửa sổ chữ nhật.
5. **Bộ lọc số FIR:** Khẳng định đặc tính pha tuyến tính với độ trễ nhóm cố định $2.27$ ms và khả năng triệt tiêu chính xác các dải tần số mong muốn.
6. **Lượng tử hóa & Mã hóa:** Xác nhận quy luật xấp xỉ $+6$ dB cho mỗi bit bổ sung, cùng hiệu quả giảm dung lượng vượt bậc của các chuẩn nén thính giác (MP3 đạt tỷ số nén từ $5.5:1$ đến $11:1$).
